In [ ]:
import	pandas					as	pd
import	plotly.graph_objects	as	go
import	plotly.express			as	px
import	numpy					as	np
df: pd.DataFrame = pd.read_parquet('data/EWA_data.parquet')
# df = df.drop(columns = df.columns[0])

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.normalize()
df['time'] = df['timestamp'] - df['timestamp'].dt.normalize()
# df['timestamp'] = df['timestamp'].astype('int64') // (6 * 10 ** 7)
cols = ['timestamp', 'date', 'time'] + [c for c in df.columns if c not in ['timestamp', 'date', 'time']]
df = df[cols]
# df = df.set_index("timestamp")

In [ ]:
df

In [ ]:
s_slice = df[(df["date"] >= pd.Timestamp('2025-02-06', tz = df['date'].dt.tz)) & (df["date"] <= pd.Timestamp('2025-02-10', tz = df['date'].dt.tz))]["close"]
print(s_slice)

In [ ]:
df2 = df[(df["date"] >= pd.Timestamp('2025-02-06', tz = df['date'].dt.tz)) & (df["date"] >= pd.Timestamp('2025-02-10', tz = df['date'].dt.tz))]
fig = px.line(df2, x = df2.index, y = "vwap", labels = {'x': 'index', 'y' : 'vwap'})
fig.update_traces(
	customdata = df2["timestamp"],
	hovertemplate = (
	"Time: <b>%{customdata|%Y-%m-%d %H:%M}</b><br>VWA Price: %{y:.2f}<br><extra></extra>"
))
fig.show()

In [ ]:
returns: pd.Series = np.log(s_slice / s_slice.shift(1)).dropna() * 1e4
print(returns.values.mean())
returns = returns - returns.values.mean()
abs_returns: pd.Series = abs(returns)
sign: pd.Series = (returns > 0).astype('int64')
fig2 = px.line(x = abs_returns.index, y = abs_returns.values, labels = {'x': 'index', 'y': 'returns'})
fig2.show()
print(len(returns))

In [ ]:
from	statsmodels.graphics.tsaplots	import	plot_acf
# fig3 = plot_acf((s_slice).dropna(), lags = 2000)
px.line(x = sign.index, y = sign.values)

In [ ]:
acf_plot = plot_acf(abs_returns, lags = 1000)

In [ ]:
import	scipy.stats	as	stat
t_param_btc = stat.t.fit(returns)
print(t_param_btc)